# Spanish Review Sentiment Classifier

**Course:** Natural Language Processing · Unit 3 — Text Classification and Sentiment Analysis  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 4 — Naive Bayes, Text Classification, and Sentiment  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Load and inspect a Spanish product review dataset
2. Pre-process text and remove **Spanish stopwords** before vectorisation
3. Convert reviews to TF-IDF features and train three classifiers
4. Evaluate models with **classification reports** and **confusion matrices**
5. Visualise the **label distribution** to detect class imbalance

## Dataset

The `dataset_reseñas_espanol.csv` file contains product reviews in Spanish with a `sentimiento` column (`positivo` / `negativo`).


---

## 1. Environment Setup

Install dependencies **once** in your terminal:

```bash
pip install scikit-learn matplotlib nltk pandas
python -c "import nltk; nltk.download('stopwords')"
```


In [ ]:
# pip install scikit-learn matplotlib nltk pandas  # run once in terminal
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

DIR_DATA = Path.cwd() / 'data'
nltk.download('stopwords', quiet=True)


---

## 2. Load the Dataset


In [ ]:
df = pd.read_csv(DIR_DATA / 'dataset_reseñas_espanol.csv', sep=',')

# Normalise column names (strip whitespace, lowercase)
df.columns = df.columns.str.strip().str.lower()

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(df.head(5).to_string())


> **Output interpretation:** Stripping column names removes invisible leading/trailing spaces that can cause `KeyError` when accessing columns by name. Inspect the head output to confirm the `reseña` (review text) and `sentimiento` (sentiment label) columns are present and non-empty.


---

## 3. Label Distribution

Visualising the class distribution before training reveals whether the dataset is balanced or skewed.


In [ ]:
label_counts = df['sentimiento'].value_counts()
print(f'Label distribution:')
print(label_counts.to_string())

# Bar chart of class frequencies
ax = label_counts.plot(kind='bar', color=['steelblue', 'salmon'], figsize=(5, 3))
ax.set_title('Sentiment label distribution')
ax.set_xlabel('Sentiment class')
ax.set_ylabel('Number of reviews')
ax.set_xticklabels(label_counts.index, rotation=0)
plt.tight_layout()
plt.show()


> **Output interpretation:** If the bars are roughly equal, the dataset is balanced and accuracy is a meaningful metric. If one class has significantly more samples, consider using macro F1 (which weights all classes equally) or applying oversampling (`RandomOverSampler`) to the minority class before training.


---

## 4. Prepare Features and Labels


In [ ]:
# Map text labels to binary integers: positivo=1, negativo=0
label_map = {'positivo': 1, 'negativo': 0}
df['label'] = df['sentimiento'].str.lower().str.strip().map(label_map)

# Drop rows where the label could not be mapped (unexpected values)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

X = df['reseña']
y = df['label']

print(f'Total samples after cleaning: {len(df):,}')
print(f'Class distribution: {y.value_counts().to_dict()}')


---

## 5. Train / Test Split


In [ ]:
# 80% train / 20% test; stratify preserves the class ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training samples: {len(X_train):,}')
print(f'Test samples:     {len(X_test):,}')


---

## 6. TF-IDF Vectorisation with Spanish Stopwords

NLTK's Spanish stopword list removes common function words (`de`, `la`, `y`, `en`, etc.) that appear in nearly every review and carry no sentiment signal.


In [ ]:
spanish_stopwords = stopwords.words('spanish')
print(f'Spanish stopwords count: {len(spanish_stopwords)}')
print(f'Sample: {spanish_stopwords[:15]}')


In [ ]:
# Fit on training data only to prevent information leakage to IDF weights
vectorizer = TfidfVectorizer(stop_words=spanish_stopwords)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')
print(f'Training matrix: {X_train_tfidf.shape}')
density = X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
print(f'Sparsity:        {1 - density:.4%}')


> **Output interpretation:** Removing stopwords typically reduces vocabulary size by 5–15% while improving downstream classification performance, because the remaining features carry more discriminative signal. The sparse TF-IDF matrix is stored in CSR format — passing it directly to scikit-learn classifiers works without conversion.


---

## 7. Train Three Classifiers


In [ ]:
# Multinomial Naive Bayes — good baseline for sparse text features
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

# Logistic Regression — linear model with interpretable weights
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)

# Linear SVM — maximum-margin classifier; strong on high-dimensional sparse data
svm = LinearSVC(random_state=42)
svm.fit(X_train_tfidf, y_train)

print('All three classifiers trained successfully.')


---

## 8. Evaluate Performance


In [ ]:
label_names = ['Negativo', 'Positivo']

for name, model in [('Naive Bayes', nb), ('Logistic Regression', lr), ('Linear SVM', svm)]:
    y_pred = model.predict(X_test_tfidf)
    print(f'=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=label_names))


> **Output interpretation:** Compare F1-scores across the three classifiers. For binary sentiment tasks on Spanish product reviews:
>
> - Naive Bayes often scores ~80–85% F1 — fast baseline.
> - Logistic Regression typically reaches ~86–90% F1 — stronger, still interpretable.
> - Linear SVM usually matches or slightly exceeds LR — especially on smaller datasets.
>
> If precision is much lower than recall for the `Negativo` class, the model is mislabelling many neutral reviews as negative — a common issue when negative reviews use more informal or sarcastic language.


---

## 9. Confusion Matrix — Naive Bayes


In [ ]:
y_pred_nb = nb.predict(X_test_tfidf)

disp = ConfusionMatrixDisplay.from_estimator(
    nb,
    X_test_tfidf,
    y_test,
    display_labels=label_names,
    cmap='Blues',
)
disp.ax_.set_title('Confusion Matrix — Naive Bayes')
plt.tight_layout()
plt.show()


> **Output interpretation:** The confusion matrix diagonal shows correct predictions; off-diagonal cells are errors. A large top-right cell (negative predicted as positive) suggests the model is too optimistic — consider adding more negative training samples or using a higher classification threshold. A large bottom-left cell (positive predicted as negative) indicates overly cautious classification — common when negative sentiment words dominate the vocabulary.


---

## Summary

| Step | Tool | Key decision |
|---|---|---|
| Load data | `pandas.read_csv` | Check column names and encoding |
| Stopword removal | NLTK Spanish stopwords | Reduces noise; may remove domain-specific words |
| Vectorisation | `TfidfVectorizer` | fit only on train to prevent leakage |
| Classification | NB / LR / LinearSVC | Compare F1; choose based on speed vs accuracy |
| Evaluation | classification report + confusion matrix | Use macro F1 for imbalanced classes |
